# 01 — Explorer une IRM : le format NIfTI, l'affine, les orientations

Ce notebook accompagne l'étape 1.4 de la [ROADMAP](../ROADMAP.md). Il ne produit rien pour le pipeline : il sert à **comprendre** les données avant d'écrire `utils/io.py`.

| Section | Notion |
|---|---|
| 1 | Télécharger deux sujets de deux scanners différents |
| 2 | Lire un en-tête NIfTI : forme, voxels, type, orientation, qform/sform |
| 3 | Décomposer la matrice affine (tailles de voxel, directions, inclinaison des coupes) |
| 4 | Passer d'un voxel à des millimètres, puis à un voxel d'une autre image |
| 5 | Afficher correctement des coupes (et voir ce qui se passe si on ignore l'orientation) |
| 6 | La vérité terrain : labels, volumes, superposition |
| 7 | Les intensités ne sont pas comparables d'un scanner à l'autre |
| 8 | Le champ de biais |
| 9 | Rééchantillonner n'est pas recaler |
| 10 | Pourquoi on n'interpole jamais un masque linéairement |
| 11 | Types de données et mémoire |
| 12 | Ce que `utils/io.py` doit garantir |

Exécution : `Run All` dans VS Code (noyau `.venv`). Environ 50 Mo téléchargés, moins de 2 Go de RAM.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import requests
from matplotlib.colors import ListedColormap
from nibabel.processing import resample_from_to
from skimage.filters import threshold_otsu

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.dpi"] = 110
LABEL_CMAP = ListedColormap(["black", "red", "deepskyblue"])   # 0 fond, 1 HSB (rouge), 2 autre pathologie (bleu)
print("racine du projet :", ROOT)

## 1. Télécharger deux sujets

Deux sujets d'entraînement de **deux scanners différents**, pour pouvoir les comparer :
- `training/Utrecht/0` : Philips 3T, FLAIR acquise en **2D** (coupes de 3 mm) ;
- `training/Amsterdam/GE3T/100` : GE 3T, FLAIR acquise en **3D** puis rééchantillonnée en coupes de 3 mm par les organisateurs.

Les fichiers sont rangés là où le futur module `data/download.py` les mettra (`data/raw/...`, ignoré par Git). Le téléchargement n'a lieu qu'une fois.

In [ ]:
API = "https://dataverse.nl/api/datasets/:persistentId/?persistentId=doi:10.34894/AECRSD"
FILE_URL = "https://dataverse.nl/api/access/datafile/{}"
SUBJECTS = {"utrecht": "training/Utrecht/0", "amsterdam": "training/Amsterdam/GE3T/100"}
WANT = {"orig/FLAIR.nii.gz", "orig/3DT1.nii.gz", "orig/T1.nii.gz", "pre/FLAIR.nii.gz", "wmh.nii.gz"}

files = requests.get(API, timeout=60).json()["data"]["latestVersion"]["files"]
for subj in SUBJECTS.values():
    for f in files:
        d = f.get("directoryLabel") or ""
        if d != subj and not d.startswith(subj + "/"):
            continue
        rel = f"{d[len(subj):].strip('/')}/{f['dataFile']['filename']}".strip("/")
        out = RAW / subj / rel
        if rel not in WANT or out.exists():
            continue
        out.parent.mkdir(parents=True, exist_ok=True)
        r = requests.get(FILE_URL.format(f["dataFile"]["id"]), timeout=300)
        r.raise_for_status()
        out.write_bytes(r.content)
        print("téléchargé :", out.relative_to(ROOT))
print("données prêtes")

In [ ]:
def load_subject(subj):
    base = RAW / subj
    return {
        "flair": nib.load(base / "orig/FLAIR.nii.gz"),          # FLAIR brute
        "t1_3d": nib.load(base / "orig/3DT1.nii.gz"),           # T1 3D brute (visage effacé)
        "t1_elastix": nib.load(base / "orig/T1.nii.gz"),        # T1 alignée sur la FLAIR par les organisateurs
        "flair_pre": nib.load(base / "pre/FLAIR.nii.gz"),       # FLAIR corrigée du biais (SPM12)
        "wmh": nib.load(base / "wmh.nii.gz"),                   # vérité terrain, grille FLAIR
    }

U = load_subject(SUBJECTS["utrecht"])
A = load_subject(SUBJECTS["amsterdam"])

## 2. Lire un en-tête NIfTI

`nib.load` ne lit que l'**en-tête** : le tableau de voxels n'est chargé qu'à la première demande (lecture paresseuse). Pour chaque fichier :
- `shape` : forme du tableau ;
- `voxel_mm` : taille des voxels (`header.get_zooms()`) ;
- `axcodes` : vers où va chaque axe du tableau quand l'indice augmente (calculé depuis l'affine) ;
- `dtype_disk` : type stocké sur le disque ;
- `qform_code` / `sform_code` : validité des deux affines (0 = non définie) et si elles coïncident ;
- la mémoire occupée selon le type.

In [ ]:
def describe(img):
    h = img.header
    q, qc = h.get_qform(coded=True)
    s, sc = h.get_sform(coded=True)
    n = int(np.prod(img.shape))
    return {
        "shape": img.shape,
        "voxel_mm": tuple(round(float(z), 2) for z in h.get_zooms()[:3]),
        "axcodes": "".join(nib.aff2axcodes(img.affine)),
        "dtype_disk": str(img.get_data_dtype()),
        "qform_code": int(qc),
        "sform_code": int(sc),
        "qform == sform": bool(qc and sc and np.allclose(q, s, atol=1e-3)),
        "Mo (type disque)": round(n * img.get_data_dtype().itemsize / 1e6, 1),
        "Mo (float64)": round(n * 8 / 1e6, 1),
    }

pd.concat({"Utrecht": pd.DataFrame({k: describe(v) for k, v in U.items()}).T,
           "Amsterdam": pd.DataFrame({k: describe(v) for k, v in A.items()}).T})

**À observer**
- La T1 3D (`PIR`) et la FLAIR (`LPS`) n'ont ni la même forme, ni les mêmes voxels, ni le même ordre d'axes : ce sont deux tableaux sans rapport direct.
- `t1_elastix`, `flair_pre` et `wmh` ont exactement la forme et les voxels de la FLAIR : ils sont **sur sa grille**.
- `wmh` est stocké en `float32` alors qu'il ne contient que des labels entiers.
- Les voxels FLAIR d'Amsterdam ne sont pas ceux du readme : ce sont des fichiers rééchantillonnés (d'où `flair_acquisition` dans le config).

## 3. Décomposer la matrice affine

L'affine `A` relie un indice de voxel `(i, j, k)` à une position `(x, y, z)` en mm, dans le repère **RAS+** de nibabel (x vers la droite du patient, y vers l'avant, z vers le haut) :

$$\begin{pmatrix}x\\y\\z\\1\end{pmatrix} = A \begin{pmatrix}i\\j\\k\\1\end{pmatrix}$$

- La **norme de chaque colonne** du bloc 3×3 est la taille du voxel le long de cet axe.
- La **colonne normalisée** est la direction de cet axe dans le monde.
- La **dernière colonne** est la position du voxel `(0, 0, 0)`.

In [ ]:
aff = U["flair"].affine
print("Affine de la FLAIR (Utrecht 0) :\n", aff)

M = aff[:3, :3]
voxel_sizes = np.linalg.norm(M, axis=0)
directions = M / voxel_sizes
print("\ntailles de voxel (mm), lues dans l'affine :", voxel_sizes)
print("en-tête (pixdim)                          :", np.array(U["flair"].header.get_zooms()[:3]))
print("\ndirection de chaque axe du tableau (colonnes i, j, k), en RAS :\n", directions)
print("\norigine : le voxel (0,0,0) est en", aff[:3, 3], "mm")

tilt = np.degrees(np.arccos(abs(directions[2, 2])))
print(f"\nl'axe des coupes (k) est incliné de {tilt:.1f}° par rapport à l'axe tête-pieds du scanner")

**Lecture**
- Directions proches de `-1` sur l'axe x pour `i` : l'indice `i` va vers la **gauche** (x décroît), d'où le `L` de `LPS`.
- Les termes hors diagonale traduisent des coupes **obliques** : à l'acquisition, le manipulateur aligne les coupes sur l'anatomie, pas sur les axes de l'aimant. Les codes d'orientation (`LPS`) ne sont qu'une approximation de ces directions.

## 4. D'un voxel à des millimètres, puis à un voxel d'une autre image

Les deux images ne communiquent **que** par l'espace physique :

`voxel FLAIR → (affine FLAIR) → mm → (inverse de l'affine T1) → voxel T1`

In [ ]:
flair, t1 = U["flair"], U["t1_3d"]
ijk_flair = (np.array(flair.shape) - 1) / 2                 # voxel central de la FLAIR
xyz = flair.affine @ np.append(ijk_flair, 1)                # -> millimètres (RAS)
ijk_t1 = np.linalg.inv(t1.affine) @ xyz                     # -> voxel de la T1 3D

print("voxel FLAIR       :", ijk_flair)
print("position (mm, RAS):", xyz[:3])
print("voxel T1 3D       :", ijk_t1[:3])
print("\nLe voxel T1 n'est pas entier : la valeur de la T1 à cet endroit doit être interpolée.")
print("C'est exactement ce que fait un rééchantillonnage, pour chaque voxel de la grille cible.")

## 5. Afficher correctement des coupes

### 5a. Ce qui se passe si on ignore l'orientation

On prend « la coupe du milieu sur le 3e axe » des deux tableaux bruts :

In [ ]:
f = U["flair"].get_fdata(dtype=np.float32)
t = U["t1_3d"].get_fdata(dtype=np.float32)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(f[:, :, f.shape[2] // 2], cmap="gray"); ax[0].set_title("FLAIR[:, :, milieu]  (LPS)")
ax[1].imshow(t[:, :, t.shape[2] // 2], cmap="gray"); ax[1].set_title("T1 3D[:, :, milieu]  (PIR)")
for a in ax: a.axis("off")
plt.show()

La même opération donne une coupe **axiale** (tournée) pour la FLAIR et une coupe **sagittale** pour la T1. Rien ne signale l'erreur.

### 5b. Affichage correct

Pour l'**affichage uniquement**, on réordonne le tableau en orientation canonique `RAS` (`nib.as_closest_canonical` : des permutations et retournements d'axes, sans interpolation ni perte). Puis :
- `imshow(coupe.T, origin="lower")` : axe horizontal = 1er axe de la coupe, axe vertical = 2e ;
- `aspect = taille voxel verticale / taille voxel horizontale` : sans ça, une FLAIR à coupes de 3 mm paraît écrasée d'un facteur 3 ;
- convention **neurologique** : la droite du patient est à droite de l'écran (la radiologie fait souvent l'inverse).

On ne réoriente **jamais** les fichiers du pipeline : la FLAIR doit rester dans sa grille native pour l'évaluation.

In [ ]:
def show_ortho(img, mask=None, ijk=None, title=""):
    can = nib.as_closest_canonical(img)
    vol = can.get_fdata(dtype=np.float32)
    zooms = can.header.get_zooms()[:3]
    i, j, k = (int(round(v)) for v in (np.array(vol.shape) // 2 if ijk is None else ijk))
    msk = None if mask is None else np.rint(nib.as_closest_canonical(mask).get_fdata()).astype(np.uint8)
    vmax = np.percentile(vol[vol > 0], 99.5)
    views = [("axiale", np.s_[:, :, k], (0, 1), (i, j), ("D", "G")),
             ("coronale", np.s_[:, j, :], (0, 2), (i, k), ("D", "G")),
             ("sagittale", np.s_[i, :, :], (1, 2), (j, k), ("A", "P"))]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
    for ax, (name, sl, (a, b), (cx, cy), (right, left)) in zip(axes, views):
        asp = zooms[b] / zooms[a]
        ax.imshow(vol[sl].T, origin="lower", cmap="gray", vmin=0, vmax=vmax, aspect=asp)
        if msk is not None:
            m = msk[sl].T
            ax.imshow(np.ma.masked_where(m == 0, m), origin="lower", cmap=LABEL_CMAP, vmin=0, vmax=2,
                      alpha=0.7, aspect=asp, interpolation="nearest")
        ax.axvline(cx, color="yellow", lw=0.5); ax.axhline(cy, color="yellow", lw=0.5)
        ax.set_title(f"{name}  (voxels {zooms[a]:.2f} x {zooms[b]:.2f} mm)", fontsize=9)
        ax.text(0.98, 0.5, right if name != "sagittale" else "P", transform=ax.transAxes, color="w", ha="right")
        ax.text(0.02, 0.5, left if name != "sagittale" else "A", transform=ax.transAxes, color="w")
        ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

def world_to_canonical_ijk(img, xyz):
    return (np.linalg.inv(nib.as_closest_canonical(img).affine) @ np.append(xyz[:3], 1))[:3]

show_ortho(U["flair"], ijk=world_to_canonical_ijk(U["flair"], xyz), title="FLAIR — croix au point central (mm)")
show_ortho(U["t1_3d"], ijk=world_to_canonical_ijk(U["t1_3d"], xyz), title="T1 3D — même point physique")

La croix tombe sur **le même endroit anatomique** dans deux tableaux de formes et d'orientations différentes : c'est l'affine qui fait le lien. Remarquer aussi la différence de résolution verticale (coupes de 3 mm pour la FLAIR, 1 mm pour la T1).

**Pourquoi les coupes « axiales » ne montrent-elles pas le même niveau ?** `as_closest_canonical` ne fait que permuter et retourner les axes : il ne supprime pas l'inclinaison. La coupe « axiale » de la FLAIR est donc le plan **incliné de 25°** de l'acquisition, tandis que celle de la T1 est (presque) horizontale. Les deux plans ne se croisent qu'au point de la croix : ailleurs, ils traversent des niveaux différents du cerveau. Pour obtenir des coupes strictement comparables, il faut **rééchantillonner** une image dans la grille de l'autre (section 9).

## 6. La vérité terrain

`wmh.nii.gz` : 0 = fond, 1 = HSB, 2 = autre pathologie (ignorée à l'évaluation). Volume d'un label = nombre de voxels × volume d'un voxel.

In [ ]:
for name, S in [("Utrecht 0", U), ("Amsterdam 100", A)]:
    w = S["wmh"]
    raw = w.get_fdata()
    lab = np.rint(raw).astype(np.uint8)
    vox_ml = float(np.prod(w.header.get_zooms()[:3])) / 1000
    same_grid = w.shape == S["flair"].shape and np.allclose(w.affine, S["flair"].affine, atol=1e-4)
    print(f"{name}: stocké en {w.get_data_dtype()}, valeurs uniques {np.unique(raw)}, "
          f"entières : {np.array_equal(raw, lab)}, même grille que la FLAIR : {same_grid}")
    for v, lname in [(1, "HSB"), (2, "autre pathologie")]:
        n = int((lab == v).sum())
        print(f"   {lname:17s}: {n:6d} voxels = {n * vox_ml:5.1f} ml")

In [ ]:
def mosaic(img, mask, n=7, title=""):
    can = nib.as_closest_canonical(img)
    vol = can.get_fdata(dtype=np.float32)
    lab = np.rint(nib.as_closest_canonical(mask).get_fdata()).astype(np.uint8)
    zooms = can.header.get_zooms()[:3]
    ks = np.sort(np.argsort((lab == 1).sum(axis=(0, 1)))[::-1][:n])   # coupes les plus chargées en HSB
    vmax = np.percentile(vol[vol > 0], 99.5)
    fig, axes = plt.subplots(2, n, figsize=(2.1 * n, 4.6))
    for c, k in enumerate(ks):
        for r in range(2):
            ax = axes[r, c]
            ax.imshow(vol[:, :, k].T, origin="lower", cmap="gray", vmin=0, vmax=vmax, aspect=zooms[1] / zooms[0])
            if r == 1:
                m = lab[:, :, k].T
                ax.imshow(np.ma.masked_where(m == 0, m), origin="lower", cmap=LABEL_CMAP, vmin=0, vmax=2,
                          alpha=0.7, aspect=zooms[1] / zooms[0], interpolation="nearest")
            ax.set_title(f"coupe {k}" if r == 0 else "", fontsize=8); ax.axis("off")
    fig.suptitle(title + "  (rouge : HSB, bleu : autre pathologie)"); plt.tight_layout(); plt.show()

mosaic(U["flair"], U["wmh"], title="Utrecht 0 — FLAIR")
mosaic(A["flair"], A["wmh"], title="Amsterdam 100 — FLAIR")

Les HSB sont les plages **blanches** autour des ventricules (périventriculaires) et dans la substance blanche profonde. Les ventricules eux-mêmes sont noirs : c'est l'effet de l'inversion FLAIR qui annule le LCR.

## 7. Les intensités ne sont pas comparables d'un scanner à l'autre

Premier-plan grossier par seuil d'Otsu (sépare automatiquement « fond noir » et « tête »), puis histogrammes des intensités de la FLAIR **brute** des deux sujets, et après une normalisation z-score (moyenne 0, écart-type 1 dans le premier plan).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, S in [("Utrecht (Philips 3T)", U), ("Amsterdam (GE 3T)", A)]:
    v = S["flair"].get_fdata(dtype=np.float32)
    lab = np.rint(S["wmh"].get_fdata()).astype(np.uint8)
    fg = v > threshold_otsu(v)
    z = (v - v[fg].mean()) / v[fg].std()
    axes[0].hist(v[fg], bins=200, histtype="step", density=True, label=name)
    axes[1].hist(z[fg], bins=200, histtype="step", density=True, label=name)
    print(f"{name:22s} brut : p1/p50/p99 = {np.percentile(v[fg], [1, 50, 99]).round(0)}, médiane HSB = {np.median(v[lab == 1]):.0f}"
          f" | z-score : médiane HSB = {np.median(z[lab == 1]):.2f}")
axes[0].set_title("FLAIR brute (unités arbitraires)"); axes[1].set_title("après z-score dans le premier plan")
for a in axes: a.legend(); a.set_yticks([])
plt.show()

Les échelles brutes n'ont rien à voir : un seuil fixé sur un scanner n'a aucun sens sur l'autre. Une normalisation simple rapproche les distributions sans les rendre identiques. C'est le point de départ de l'harmonisation (Phase 8) et de la méthode de seuillage (M3).

Dans un même sujet, les HSB sont-elles séparables du reste par l'intensité seule ?

In [ ]:
v = U["flair"].get_fdata(dtype=np.float32)
lab = np.rint(U["wmh"].get_fdata()).astype(np.uint8)
fg = v > threshold_otsu(v)
plt.figure(figsize=(7, 3.5))
plt.hist(v[fg & (lab == 0)], bins=200, density=True, alpha=0.6, label="reste du premier plan")
plt.hist(v[lab == 1], bins=100, density=True, alpha=0.6, label="HSB")
plt.legend(); plt.title("Utrecht 0 — intensités FLAIR"); plt.yticks([]); plt.show()

Les HSB sont plus claires en moyenne, mais les distributions **se chevauchent** : un seuil seul confondra des lésions pâles avec du tissu sain (et inversement le cortex, souvent un peu clair en FLAIR). C'est la limite attendue de M3, et ce qui justifie un modèle qui exploite le contexte (nnU-Net).

## 8. Le champ de biais

L'antenne de réception n'est pas homogène : une même substance blanche paraît plus claire près de l'antenne qu'au centre. Ce **champ de biais** est lisse et multiplicatif. Le rapport `FLAIR brute / FLAIR corrigée (pre/, SPM12)` le rend visible. Notre pipeline le corrigera avec N4 (ANTs).

In [ ]:
o, p = U["flair"], U["flair_pre"]
print("même grille :", o.shape == p.shape and np.allclose(o.affine, p.affine, atol=1e-4))
ov, pv = o.get_fdata(dtype=np.float32), p.get_fdata(dtype=np.float32)
fg = ov > threshold_otsu(ov)
ratio = np.full(ov.shape, np.nan, dtype=np.float32)
ratio[fg] = ov[fg] / np.maximum(pv[fg], 1e-3)
ratio /= np.nanmedian(ratio)

can = nib.as_closest_canonical(nib.Nifti1Image(ratio, o.affine))
r = can.get_fdata(dtype=np.float32); z = can.header.get_zooms()[:3]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
im = axes[0].imshow(r[:, :, r.shape[2] // 2].T, origin="lower", cmap="coolwarm", vmin=0.8, vmax=1.2, aspect=z[1] / z[0])
axes[1].imshow(r[:, r.shape[1] // 2, :].T, origin="lower", cmap="coolwarm", vmin=0.8, vmax=1.2, aspect=z[2] / z[0])
axes[0].set_title("axiale"); axes[1].set_title("coronale")
for a in axes: a.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8, label="brute / corrigée (normalisé)")
plt.show()

## 9. Rééchantillonner n'est pas recaler

- **Rééchantillonner** : calculer une image sur une autre grille en utilisant uniquement les affines (section 4, pour chaque voxel).
- **Recaler** : *estimer* une transformation supplémentaire, parce que la tête a bougé entre les deux acquisitions.

On rééchantillonne la T1 3D sur la grille FLAIR **avec les affines seules**, et on compare à la T1 recalée par les organisateurs (elastix).

In [ ]:
t1_resampled = resample_from_to(U["t1_3d"], U["flair"], order=1)   # affines seules, interpolation linéaire
a = t1_resampled.get_fdata(dtype=np.float32)
b = U["t1_elastix"].get_fdata(dtype=np.float32)
m = b > np.percentile(b, 50)
print("grille de sortie = grille FLAIR :", t1_resampled.shape == U["flair"].shape)
print(f"corrélation avec la T1 recalée par elastix : {np.corrcoef(a[m], b[m])[0, 1]:.3f}")

def checkerboard(x, y, n=8):
    x = x / np.percentile(x, 99); y = y / np.percentile(y, 99)
    ii, jj = np.indices(x.shape[:2])
    board = ((ii // (x.shape[0] // n) + jj // (x.shape[1] // n)) % 2).astype(bool)
    return np.where(board[..., None], x, y)

n_tiles = 8
diff = np.abs(a / np.percentile(a, 99) - b / np.percentile(b, 99))
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (title, vol, cmap, vmax) in zip(axes, [("damier : affines seules / recalage elastix", checkerboard(a, b, n_tiles), "gray", 1),
                                               ("|différence| (normalisée)", diff, "magma", 0.5)]):
    can = nib.as_closest_canonical(nib.Nifti1Image(vol.astype(np.float32), U["flair"].affine))
    c = can.get_fdata(dtype=np.float32); z = can.header.get_zooms()[:3]
    sl = c[:, :, c.shape[2] // 2].T
    ax.imshow(sl, origin="lower", cmap=cmap, vmin=0, vmax=vmax, aspect=z[1] / z[0])
    if cmap == "gray":   # tile borders
        for t in range(1, n_tiles):
            ax.axvline(t * sl.shape[1] / n_tiles, color="yellow", lw=0.4)
            ax.axhline(t * sl.shape[0] / n_tiles, color="yellow", lw=0.4)
    ax.set_title(title, fontsize=9); ax.axis("off")
plt.show()

Les cases alternent les deux images : si les contours (ventricules, sillons) se raccordent d'une case à l'autre, les images sont alignées. La carte des différences montre où elles divergent : surtout aux bords des structures, signe d'un petit décalage. Même sans recalage, les affines alignent déjà presque les images (même séance, même scanner). Le recalage corrige le petit mouvement résiduel : c'est ce que mesurera la corrélation au contrôle qualité de l'étape 3.2.

## 10. Pourquoi on n'interpole jamais un masque linéairement

On rééchantillonne `wmh.nii.gz` sur une grille **décalée d'un demi-voxel**, comme après un recalage :
- au **plus proche voisin** (`order=0`) ;
- en **linéaire** (`order=1`), puis arrondi.

Entre un voxel « fond » (0) et un voxel « autre pathologie » (2), la moyenne vaut 1, c'est-à-dire « HSB ». Pour isoler exactement ces lésions inventées, on interpole aussi l'indicatrice des vraies HSB : là où elle vaut 0, aucune vraie HSB n'est à proximité.

In [ ]:
w = U["wmh"]
lab = np.rint(w.get_fdata()).astype(np.uint8)
shift = np.eye(4); shift[:3, 3] = [0.5, 0.5, 0.0]                      # demi-voxel en i et j
target = (w.shape, w.affine @ shift)

lab_img = nib.Nifti1Image(lab.astype(np.float32), w.affine)
nn = np.rint(resample_from_to(lab_img, target, order=0).get_fdata()).astype(np.uint8)
lin = np.rint(resample_from_to(lab_img, target, order=1).get_fdata()).astype(np.uint8)
true_wmh = resample_from_to(nib.Nifti1Image((lab == 1).astype(np.float32), w.affine), target, order=1).get_fdata()
invented = (lin == 1) & (true_wmh < 1e-3)

print("labels après plus proche voisin   :", np.unique(nn))
print("labels après linéaire + arrondi    :", np.unique(lin))
print("voxels « HSB » inventés entre 0 et 2 :", int(invented.sum()),
      f"(sur {int((lab == 1).sum())} voxels HSB réels)")

In [ ]:
if invented.any():
    k = int(np.argmax(invented.sum(axis=(0, 1))))
    ii, jj = np.nonzero(invented[:, :, k])
    ci, cj = int(np.median(ii)), int(np.median(jj))
    win = np.s_[max(ci - 15, 0):ci + 15, max(cj - 15, 0):cj + 15, k]
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
    for ax, (name, arr) in zip(axes, [("original", lab), ("plus proche voisin", nn), ("linéaire + arrondi", lin)]):
        sl = arr[win].T
        ax.imshow(sl, origin="lower", cmap=LABEL_CMAP, vmin=0, vmax=2, interpolation="nearest")
        ax.set_title(name); ax.axis("off")
    fig.suptitle("zoom (grille native, axes tableau) : rouge = HSB, bleu = autre pathologie"); plt.show()

In [ ]:
# Même avec un masque binaire, le choix d'interpolation change le volume (notre biomarqueur)
binary = nib.Nifti1Image((lab == 1).astype(np.float32), w.affine)
vox_ml = float(np.prod(w.header.get_zooms()[:3])) / 1000
b_nn = resample_from_to(binary, target, order=0).get_fdata()
b_lin = resample_from_to(binary, target, order=1).get_fdata()
print(f"volume HSB original               : {(lab == 1).sum() * vox_ml:6.2f} ml")
print(f"plus proche voisin                : {(b_nn > 0.5).sum() * vox_ml:6.2f} ml")
print(f"linéaire, seuil 0,5               : {(b_lin >= 0.5).sum() * vox_ml:6.2f} ml")
print(f"linéaire, tout voxel > 0          : {(b_lin > 0).sum() * vox_ml:6.2f} ml   <- les bords gonflent")

## 11. Types de données et mémoire

Le type **sur le disque** (ici `uint16`) n'est pas le type **renvoyé** : `get_fdata()` convertit toujours en flottants (`float64` par défaut). nibabel garde de plus ce tableau en cache dans l'objet image.

In [ ]:
img = nib.load(RAW / SUBJECTS["utrecht"] / "orig/3DT1.nii.gz")
for label, arr in [("np.asanyarray(img.dataobj)  (type stocké)", np.asanyarray(img.dataobj)),
                   ("img.get_fdata(dtype=np.float32)", img.get_fdata(dtype=np.float32)),
                   ("img.get_fdata()  (défaut)", img.get_fdata())]:
    print(f"{label:45s} {str(arr.dtype):8s} {arr.nbytes / 1e6:6.0f} Mo")
img.uncache()   # libère le tableau mis en cache par get_fdata()

## 12. Ce que `utils/io.py` doit garantir

| Constat dans ce notebook | Garde-fou dans `io.py` |
|---|---|
| `get_fdata()` renvoie du float64 et le met en cache | `load_data(path, dtype=np.float32)` |
| `wmh.nii.gz` est en float32 | `load_labels(path, allowed=(0, 1, 2))` : arrondi, uint8, erreur si valeur inattendue |
| Deux tableaux de même forme peuvent ne pas être sur la même grille | `same_grid(a, b)` : forme **et** affine, avant tout calcul voxel à voxel |
| Sauvegarder sans la bonne affine déplace l'image en silence | `save_like(data, reference, path)` : affine et en-tête de la référence, vérification de forme |
| qform et sform peuvent diverger | `describe(img)` expose les deux codes et leur cohérence |
| Les masques ne s'interpolent pas linéairement | règle reprise dans `preproc/register.py` (plus proche voisin ou `genericLabel`) |

**Questions de contrôle** (réponses dans les sorties ci-dessus) :
1. Pourquoi `FLAIR[:, :, milieu]` et `T1[:, :, milieu]` ne montrent-ils pas le même plan ?
2. De combien de degrés sont inclinées les coupes de la FLAIR d'Utrecht 0 ?
3. Quel est le volume de HSB d'Utrecht 0 ? (environ 24,2 ml)
4. Quelle est la différence entre rééchantillonner et recaler ?
5. Combien de voxels « HSB » une interpolation linéaire invente-t-elle sur ce sujet ?